# COVID-19 Global Data Analysis: Data Exploration
## Part 1: Dataset Overview, Schema Discovery, and Data Profiling

**Lead Data Engineer & Curator:** **Himanshu Bagde** ([GitHub: @Himanshubagde11](https://github.com/Himanshubagde11))  
**Primary Surveillance Source:** Our World in Data / WHO / Johns Hopkins CSSE  
**Project:** COVID-19 Global Data Analysis & Trend Visualization

This notebook performs initial exploratory profiling on the raw COVID-19 global dataset.

### Objectives:
1. Load the raw dataset and inspect dimensional shape.
2. Examine column inventory and underlying data types.
3. Quantify missing value rates and patterns across features.
4. Detect duplicate records across spatial and temporal keys.
5. Determine global temporal bounds and country/territory coverage.



In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Add project root to path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.ingestion.load_data import load_csv, get_dataset_summary

DATA_PATH = ROOT_DIR / "data" / "raw" / "owid-covid-data.csv"
print(f"Loading raw dataset from: {DATA_PATH}")



### 1. Load Raw Dataset and Inspect Shape


In [ ]:
df_raw = load_csv(DATA_PATH)
print(f"Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head(5)



### 2. Column Inventory & Data Types


In [ ]:
summary = get_dataset_summary(df_raw)
print(f"Memory footprint: {summary['memory_usage_mb']:.2f} MB")
dtypes_df = pd.DataFrame(list(summary['data_types'].items()), columns=['Column', 'Data_Type'])
dtypes_df.head(20)



### 3. Missing Value Profiling


In [ ]:
null_df = pd.DataFrame({
    'Column': list(summary['null_counts'].keys()),
    'Null_Count': list(summary['null_counts'].values()),
    'Null_Pct': list(summary['null_percentages'].values())
}).sort_values('Null_Pct', ascending=False)

null_df.head(25)



### 4. Duplicate Record Analysis


In [ ]:
c_col = 'location' if 'location' in df_raw.columns else 'country'
d_col = 'date'

dup_keys = df_raw.duplicated(subset=[c_col, d_col]).sum()
print(f"Duplicate ({c_col}, {d_col}) pairs: {dup_keys:,}")



### 5. Geographic and Temporal Coverage


In [ ]:
distinct_locations = df_raw[c_col].nunique()
min_date = df_raw[d_col].min()
max_date = df_raw[d_col].max()

print(f"Distinct Reporting Locations: {distinct_locations}")
print(f"Temporal Window: {min_date} to {max_date}")

